# Kaggle survey v3 — Y62 phần 1: tệp số liệu zero-shot k = 2 (Mục 7.4)

Yêu cầu 09/11/2026, Việc 1. **Không chạy lại E7**: chỉ đọc dữ liệu sau QC và kết quả E7 (commit `9b2d6a5`), gọi
`scripts/run_e7_k2_evidence.py`. Thời gian chạy: vài phút, chỉ cần CPU.

## Input cần thêm (Add Input)

| Input | Nội dung | Kiểm |
|---|---|---|
| `experiment-data-v3` | bản 29/10 sau QC (TCGA + 4 cohort GEO + `fewshot_splits.csv`) | Gate 1 |
| output việc `e7` | `e7_results.zip` (commit 9b2d6a5) | SHA-256 16 ký tự `0254a5edb383841a` |
| output việc `tables` | `tables_results.zip` bản [v03-11] | SHA-256 16 ký tự `a03ebe3d56743fa2` (chỉ lấy `e7_transfer_table.csv` để đối chiếu) |

## Đầu ra

`/kaggle/working/Review-9-11.zip` gồm `results/survey_v2/E7_k2_evidence/`:
`e7_k2_gene_directions.csv`, `e7_k2_zero_shot_per_run.csv`, `e7_k2_zero_shot_summary.csv` (+ `_by_n_t`),
`e7_k2_source_coefficient.csv`, `e7_k2_checks.json`, `config.json` và `00_DOC-TRUOC.md` (commit, SHA-256 16 ký tự, thời gian chạy).

## Nghiệm thu

`config.json`: `traceable = true`. `e7_k2_checks.json`: `reproduces_e7_per_run = true` (mô hình fit lại cho đúng AUROC của E7).
Các cờ còn lại (`panel_genes_higher_in_lusc_all_cohorts`, `phi_auroc_tcga`, `n_runs_below_0_5`, `coef_phi_negative_all`,
`zero_shot_mean_matches_expected_3dp`) là kết quả cần báo, **không chỉnh số cho khớp kỳ vọng**: lệch thì ghi chú trong `00_DOC-TRUOC.md`.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')   # sửa theo slug Kaggle Input của bạn
E7_RESULTS_ZIP = None        # None = tự tìm e7_results.zip dưới /kaggle/input; hoặc Path('/kaggle/input/kltn-e7/e7_results.zip')
TABLES_RESULTS_ZIP = None    # None = tự tìm tables_results.zip dưới /kaggle/input
EXPECTED_SHA16 = {'e7_results.zip': '0254a5edb383841a', 'tables_results.zip': 'a03ebe3d56743fa2'}
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
REPO_COMMIT = None           # commit đã push có scripts/run_e7_k2_evidence.py; None = đầu nhánh
RUN_TESTS = True

WORK_ROOT = Path('/kaggle/working/kltn_data')
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
HANDOVER = Path('/kaggle/working/Review-9-11')
T0 = time.time()


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def find_input(name, given):
    if given is not None:
        return Path(given)
    hits = sorted(Path('/kaggle/input').rglob(name))
    assert hits, f'Không thấy {name} dưới /kaggle/input — Add Input output của việc tương ứng'
    if len(hits) > 1:
        print(f'Nhiều {name}: {hits} -> chọn bản có SHA-256 đúng')
        good = [h for h in hits if sha256(h)[:16] == EXPECTED_SHA16[name]]
        hits = good or hits
    return hits[0]

In [ ]:
# Gate 1 — dữ liệu sau QC đủ cho 5 bộ.
DATASETS = {'tcga_luad_lusc_primary_tumor': 'experiment-data-perm', 'gse10245': 'experiment-data-geo-target',
            'gse30219': 'experiment-data-geo-target', 'gse37745': 'experiment-data-geo-target', 'gse42127': 'experiment-data-geo-target'}
assert INPUT_V3_DATASET.is_dir(), f'Không thấy Kaggle Input: {INPUT_V3_DATASET}'
for name, branch in DATASETS.items():
    d = INPUT_V3_DATASET / branch / name
    for f in ('expression.h5', 'metadata.csv', f'excluded_samples_{name}.csv'):
        assert (d / f).is_file(), f'Thiếu {name}/{f} — Kaggle Input chưa phải bản 29/10 sau QC'
    if branch == 'experiment-data-geo-target':
        header = (d / 'fewshot_splits.csv').open(encoding='utf-8-sig').readline()
        assert 'subset' in header, f'{name}/fewshot_splits.csv là bản cũ (thiếu cột subset)'
E7_ZIP = find_input('e7_results.zip', E7_RESULTS_ZIP)
TABLES_ZIP = find_input('tables_results.zip', TABLES_RESULTS_ZIP)
INPUT_SHA = {}
for z in (E7_ZIP, TABLES_ZIP):
    INPUT_SHA[z.name] = sha256(z)
    ok = INPUT_SHA[z.name][:16] == EXPECTED_SHA16[z.name]
    print(f'{z}  {INPUT_SHA[z.name][:16]}  {"OK" if ok else "KHÁC KỲ VỌNG " + EXPECTED_SHA16[z.name]}')
    assert ok, f'{z.name} không phải bản chính thức — kiểm lại Input'
print('INPUT_OK')

In [ ]:
# Gate 2 — clone đúng commit đã push.
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
if REPO_COMMIT:
    subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
pushed = subprocess.run(['git', 'branch', '-r', '--contains', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
assert pushed, f'{commit_hash} không có trên remote — kết quả sẽ không truy vết được'
assert (REPO_DIR / 'scripts' / 'run_e7_k2_evidence.py').is_file(), 'commit này chưa có scripts/run_e7_k2_evidence.py'
for mod in ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl'):
    print(f'{mod:14s}', getattr(__import__(mod), '__version__', 'ok'))
print('COMMIT', commit_hash)

In [ ]:
# Gate 3 — liên kết dữ liệu; giải nén E7 và e7_transfer_table.csv (chỉ các phần cần, không để zip này ghi đè zip kia).
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    link = WORK_ROOT / folder
    if not link.exists() and (INPUT_V3_DATASET / folder).exists():
        link.symlink_to(INPUT_V3_DATASET / folder, target_is_directory=True)
with zipfile.ZipFile(E7_ZIP) as zf:
    zf.extractall(WORK_ROOT, members=[m for m in zf.namelist() if m.startswith('results/survey_v2/E7/')])
with zipfile.ZipFile(TABLES_ZIP) as zf:
    zf.extract('results/survey_v2/tables/e7_transfer_table.csv', WORK_ROOT)
for c in ('gse10245', 'gse30219', 'gse37745', 'gse42127'):
    assert (WORK_ROOT / 'results/survey_v2/E7' / c / 'e7_per_run.csv').is_file(), f'thiếu E7/{c}/e7_per_run.csv'
os.environ.update({'KLTN_DATA_ROOT': str(WORK_ROOT), 'KLTN_STRICT_N': '1', 'PYTHONUTF8': '1'})
os.environ.pop('KLTN_ALLOW_UNPUSHED', None)
print('RESTORED_OK')

In [ ]:
# Kiểm tra nhanh môi trường (test đơn vị, không cần dữ liệu thật).
if RUN_TESTS:
    r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '-m', 'not slow', '-p', 'no:cacheprovider', '-x'], cwd=REPO_DIR)
    assert r.returncode == 0, 'pytest không đạt trên Kaggle — không chạy'

In [ ]:
# Chạy script.
t_run = time.time()
r = subprocess.run([sys.executable, 'scripts/run_e7_k2_evidence.py'], cwd=REPO_DIR)
RUN_SECONDS = time.time() - t_run
assert r.returncode == 0, f'script lỗi (mã {r.returncode})'
OUT = WORK_ROOT / 'results/survey_v2/E7_k2_evidence'
cfg = json.loads((OUT / 'config.json').read_text(encoding='utf-8'))
checks = json.loads((OUT / 'e7_k2_checks.json').read_text(encoding='utf-8'))
print('traceable =', cfg.get('traceable'), cfg.get('traceable_errors'))
print('reproduces_e7_per_run =', checks.get('reproduces_e7_per_run'))
assert cfg.get('traceable') is True, 'kết quả không truy vết được — không được trích'

In [ ]:
import pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 30)
for f in ('e7_k2_gene_directions.csv', 'e7_k2_zero_shot_summary.csv', 'e7_k2_zero_shot_summary_by_n_t.csv', 'e7_k2_source_coefficient.csv'):
    print('=====', f)
    display(pd.read_csv(OUT / f))

In [ ]:
# Gói bàn giao: Review-9-11/ + 00_DOC-TRUOC.md -> /kaggle/working/Review-9-11.zip
if HANDOVER.exists(): shutil.rmtree(HANDOVER)
dst = HANDOVER / 'results/survey_v2/E7_k2_evidence'
shutil.copytree(OUT, dst)
rows = []
for p in sorted(dst.iterdir()):
    rows.append(f'| `{p.name}` | `{sha256(p)[:16]}` | {p.stat().st_size} |')
flags = {k: v for k, v in checks.items() if k not in ('expected', 'source_panel_k2', 'e7_commit_hash', 'e7_traceable')}
doc = f'''# Review-9-11 — Y62 phần 1 (yêu cầu 09/11/2026, Việc 1)

- Script: `scripts/run_e7_k2_evidence.py`
- Commit: `{commit_hash}` (nhánh `{REPO_BRANCH}`, đã push); `traceable = {cfg.get("traceable")}`
- Kết quả E7 đọc vào: commit `{", ".join(sorted(set(checks["e7_commit_hash"].values())))}`
- Input: `e7_results.zip` `{INPUT_SHA["e7_results.zip"][:16]}`, `tables_results.zip` `{INPUT_SHA["tables_results.zip"][:16]}`
- Thời gian chạy script: {RUN_SECONDS:.1f} s; cả notebook: {(time.time() - T0) / 60:.1f} phút
- Chạy lúc: {time.strftime("%Y-%m-%d %H:%M:%S %Z")}
- Panel nguồn k = 2: {checks["source_panel_k2"]}

## Tệp (SHA-256 16 ký tự đầu)

| Tệp | SHA-256 | Byte |
|---|---|---|
''' + '\n'.join(rows) + f'''

## Kiểm tra (e7_k2_checks.json)

```json
{json.dumps(flags, ensure_ascii=False, indent=2)}
```

## Ghi chú

- Thang đo GSE42127 là log2 MBCB (Illumina GPL6884), không phải RMA/gcRMA như yêu cầu ghi.
- E7 dùng seed CV nội bộ riêng theo cohort đích, nên `e7_k2_source_coefficient.csv` có một mô hình nguồn cho mỗi cohort đích.
- (Đội code ghi thêm ở đây nếu số nào lệch kỳ vọng.)
'''
(HANDOVER / '00_DOC-TRUOC.md').write_text(doc, encoding='utf-8')
archive = Path('/kaggle/working/Review-9-11.zip')
if archive.exists(): archive.unlink()
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=HANDOVER.parent, base_dir=HANDOVER.name)
print(doc)
print('ARCHIVE', archive, f'{archive.stat().st_size / 1e3:.1f} KB')